# 🛒 Shopify Universal ID Checker
Mendukung tipe halaman: **collection**, **article**, dan **page**

---
| Setting | Value |
|---|---|
| Input | Upload file `.txt` berisi URL (satu per baris) |
| Output | `shopify_ids.csv` (disimpan real-time) |
| Delay | 5 detik antar request |
| Kolom output | `url`, `page_type`, `shopify_id`, `cms_url`, `status` |

## 📦 Install Dependencies

In [ ]:
!pip install requests beautifulsoup4 pandas -q

## ⚙️ Konfigurasi & Upload
Masukkan **Shopify Store Name** (slug dari URL admin) lalu upload file `.txt` berisi daftar URL.

> Contoh: URL admin `https://admin.shopify.com/store/**sme-brother**/...` → store name = `sme-brother`

In [ ]:
from google.colab import files

# ── Input Store Name ─────────────────────────────────────────
STORE_NAME = input("🏪 Masukkan Shopify store name (contoh: sme-brother): ").strip()

if not STORE_NAME:
    raise ValueError("❌ Store name tidak boleh kosong!")

STORE_NAME = STORE_NAME.strip("/")
ADMIN_BASE = f"https://admin.shopify.com/store/{STORE_NAME}"
print(f"\n✅ Store name     : {STORE_NAME}")
print(f"🔗 Admin base URL : {ADMIN_BASE}/")

# ── Upload File URL ──────────────────────────────────────────
print("\n📂 Silakan upload file .txt berisi URL Shopify...")
uploaded = files.upload()

if not uploaded:
    raise ValueError("❌ Tidak ada file yang diupload!")

INPUT_FILE  = list(uploaded.keys())[0]
OUTPUT_FILE = "shopify_ids.csv"

print(f"\n✅ File diterima  : {INPUT_FILE}")
print(f"💾 Output file    : {OUTPUT_FILE}")

## 🚀 Jalankan Scraper

In [ ]:
# ============================================================
# Shopify Universal ID Checker
# Supports  : collection, article, page
# Detection : auto dari pola URL
# Output    : url, page_type, shopify_id, cms_url, status
# Saves     : real-time ke CSV (aman jika berhenti di tengah)
# ============================================================

import re
import json
import time
import pathlib
import requests
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime

# ── Config ──────────────────────────────────────────────────
DELAY_SECONDS   = 5     # Jeda antar request (detik)
REQUEST_TIMEOUT = 30    # Timeout per request (detik)

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"
)

SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": USER_AGENT,
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.8,id;q=0.7",
    "Connection": "keep-alive",
})

# ── Logger ───────────────────────────────────────────────────
def log(level: str, msg: str):
    icons = {
        "INFO":  "ℹ️ ",
        "OK":    "✅",
        "WARN":  "⚠️ ",
        "ERROR": "❌",
        "SKIP":  "⏭️ ",
        "SAVE":  "💾",
        "DONE":  "🎉",
        "DELAY": "⏳",
        "LINK":  "🔗",
    }
    ts   = datetime.now().strftime("%H:%M:%S")
    icon = icons.get(level, "  ")
    print(f"[{ts}] {icon} [{level}] {msg}")

# ── URL Type Detector ────────────────────────────────────────
def detect_page_type(url: str) -> str:
    """
    Deteksi tipe halaman Shopify dari pola URL:
      /collections/             → collection
      /blogs/{handle}/{article} → article  (2 segment setelah /blogs/)
      /blogs/{handle}           → blog     (1 segment setelah /blogs/)
      /pages/                   → page
      lainnya                   → unknown
    """
    from urllib.parse import urlparse
    u = url.lower()
    if "/collections/" in u:
        return "collection"
    elif "/blogs/" in u:
        path_parts = [p for p in urlparse(url).path.split("/") if p]
        blogs_idx = next((i for i, p in enumerate(path_parts) if p.lower() == "blogs"), None)
        if blogs_idx is not None and len(path_parts) > blogs_idx + 2:
            return "article"
        return "blog"
    elif "/pages/" in u:
        return "page"
    return "unknown"

# ── CMS URL Builder ──────────────────────────────────────────
def build_cms_url(store_name: str, page_type: str, shopify_id: int) -> str:
    """
    Generate Shopify Admin CMS URL dinamis.

    Pola:
      collection → https://admin.shopify.com/store/{store}/collections/{id}
      article    → https://admin.shopify.com/store/{store}/content/articles/{id}
      blog       → https://admin.shopify.com/store/{store}/content/blogs/{id}
      page       → https://admin.shopify.com/store/{store}/pages/{id}
    """
    base  = f"https://admin.shopify.com/store/{store_name}"
    paths = {
        "collection": f"{base}/collections/{shopify_id}",
        "article":    f"{base}/content/articles/{shopify_id}",
        "blog":       f"{base}/content/blogs/{shopify_id}",
        "page":       f"{base}/pages/{shopify_id}",
    }
    return paths.get(page_type, "")

# ── HTML Extractor ───────────────────────────────────────────
def extract_id_from_html(html: str, expected_type: str) -> int | None:
    """
    Ekstrak numeric ID dari HTML Shopify.
    Mencoba 3 metode secara berurutan:
      1. Parse JSON var __st
      2. Regex fallback pada raw HTML
      3. HTML data-*-id attribute
    """
    # Metode 1: JSON var __st
    m = re.search(r"var\s+__st\s*=\s*(\{.*?\});", html, flags=re.DOTALL)
    if m:
        try:
            payload = json.loads(m.group(1))
            p_val = payload.get("p")
            # "blog" page type uses "blog" in __st; match directly or via rtyp
            if p_val == expected_type or expected_type == "unknown" or payload.get("rtyp") == expected_type:
                rid = payload.get("rid")
                if isinstance(rid, int):
                    return rid
                if isinstance(rid, str) and rid.isdigit():
                    return int(rid)
        except json.JSONDecodeError:
            pass

    # Metode 2: Regex fallback
    m2 = re.search(
        rf'"p"\s*:\s*"{re.escape(expected_type)}"[^}}]*?"rid"\s*:\s*"?(\d+)"?',
        html,
        flags=re.DOTALL
    )
    if m2:
        return int(m2.group(1))

    # Metode 3: HTML attribute
    soup = BeautifulSoup(html, "html.parser")
    attr_map = {
        "collection": "data-collection-id",
        "article":    "data-article-id",
        "blog":       "data-blog-id",
        "page":       "data-page-id",
    }
    attr = attr_map.get(expected_type)
    if attr:
        tag = soup.find(attrs={attr: True})
        if tag:
            val = tag.get(attr)
            if val and str(val).isdigit():
                return int(val)

    return None

# ── Main Fetcher ─────────────────────────────────────────────
def fetch_shopify_id(url: str) -> dict:
    """
    Fetch halaman dan ekstrak ID.
    Return dict: {url, page_type, shopify_id, cms_url, status}
    """
    result = {
        "url":        url,
        "page_type":  None,
        "shopify_id": None,
        "cms_url":    None,
        "status":     None,
    }

    # Skip admin URL
    if "admin.shopify.com" in url or "/admin/" in url:
        result["status"] = "skipped (admin URL)"
        return result

    # Deteksi tipe halaman dari URL
    page_type = detect_page_type(url)
    result["page_type"] = page_type

    if page_type == "unknown":
        result["status"] = "skipped (URL type unknown)"
        return result

    # HTTP Request
    # Encode non-ASCII characters in URL (e.g. Chinese/CJK slugs)
    try:
        from urllib.parse import urlsplit, urlunsplit, quote
        parts = urlsplit(url)
        encoded_url = urlunsplit(parts._replace(
            path=quote(parts.path, safe="/"),
            query=quote(parts.query, safe="=&"),
        ))
    except Exception:
        encoded_url = url

    try:
        resp = SESSION.get(encoded_url, timeout=REQUEST_TIMEOUT)
        resp.raise_for_status()
    except Exception as e:
        result["status"] = f"error ({e})"
        return result

    # Ekstrak ID
    shopify_id = extract_id_from_html(resp.text, page_type)

    if shopify_id:
        result["shopify_id"] = shopify_id
        result["cms_url"]    = build_cms_url(STORE_NAME, page_type, shopify_id)
        result["status"]     = "ok"
    else:
        snap_path = f"_snapshot_{int(time.time())}.html"
        try:
            pathlib.Path(snap_path).write_text(resp.text, encoding="utf-8")
            result["status"] = f"id not found (snapshot: {snap_path})"
        except Exception:
            result["status"] = "id not found"

    return result

# ── CSV Helpers ──────────────────────────────────────────────
CSV_COLUMNS = ["url", "page_type", "shopify_id", "cms_url", "status"]

def init_output_csv(output_path: str):
    """Inisiasi file CSV kosong dengan header di awal program."""
    pd.DataFrame(columns=CSV_COLUMNS).to_csv(output_path, index=False, encoding="utf-8")
    log("SAVE", f"Output CSV diinisiasi: {output_path}")

def append_to_csv(output_path: str, row: dict):
    """Tambahkan satu baris ke CSV secara real-time."""
    pd.DataFrame([row])[CSV_COLUMNS].to_csv(
        output_path, mode="a", header=False, index=False, encoding="utf-8"
    )

# ── Entry Point ──────────────────────────────────────────────
def main():
    try:
        with open(INPUT_FILE, "r", encoding="utf-8") as f:
            urls = [ln.strip() for ln in f if ln.strip()]
    except FileNotFoundError:
        log("ERROR", f"File tidak ditemukan: {INPUT_FILE}")
        return

    if not urls:
        log("WARN", "Tidak ada URL di file input.")
        return

    total = len(urls)
    log("INFO", f"Store    : {STORE_NAME}")
    log("INFO", f"Ditemukan {total} URL — mulai proses...")
    print("═" * 60)

    # Inisiasi CSV di awal — data aman walau program berhenti di tengah jalan
    init_output_csv(OUTPUT_FILE)

    success = 0
    failed  = 0
    skipped = 0

    for i, url in enumerate(urls, start=1):
        print(f"\n── [{i}/{total}] ──────────────────────────────────")
        log("INFO",  f"URL      : {url}")

        result = fetch_shopify_id(url)

        if result["status"] == "ok":
            log("OK",   f"Type     : {result['page_type']}")
            log("OK",   f"ID       : {result['shopify_id']}")
            log("LINK", f"CMS URL  : {result['cms_url']}")
            success += 1
        elif result["status"] and result["status"].startswith("skipped"):
            log("SKIP", f"Reason   : {result['status']}")
            skipped += 1
        else:
            log("WARN", f"Status   : {result['status']}")
            failed += 1

        append_to_csv(OUTPUT_FILE, result)
        log("SAVE", f"Baris disimpan ke {OUTPUT_FILE}")

        if i < total:
            log("DELAY", f"Menunggu {DELAY_SECONDS} detik...")
            time.sleep(DELAY_SECONDS)

    print("\n" + "═" * 60)
    log("DONE", f"Selesai! Total: {total} | ✅ OK: {success} | ⚠️  Gagal: {failed} | ⏭️  Skip: {skipped}")
    log("SAVE", f"Hasil lengkap tersimpan di: {OUTPUT_FILE}")

main()

## 💾 Download Hasil

In [ ]:
from google.colab import files
import os

if os.path.exists(OUTPUT_FILE):
    print(f"📥 Mendownload {OUTPUT_FILE}...")
    files.download(OUTPUT_FILE)
else:
    print("❌ File output belum tersedia. Jalankan cell scraper terlebih dahulu.")

## 👀 Preview Hasil

In [ ]:
import pandas as pd
import os

if os.path.exists(OUTPUT_FILE):
    df = pd.read_csv(OUTPUT_FILE)
    print(f"📊 Total rows  : {len(df)}")
    print(f"✅ Berhasil    : {(df['status'] == 'ok').sum()}")
    print(f"❌ Gagal       : {df['status'].str.startswith('error').sum()}")
    print(f"⏭️  Skip        : {df['status'].str.startswith('skipped').sum()}")
    print("\n")
    display(df)
else:
    print("❌ File output belum tersedia.")